In [1]:
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.logger import configure
import gymnasium as gym
from gymnasium import spaces
import imageio
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Circle
import tensorflow as tf
import datetime
import os



In [2]:
# Agent and opponent class
#class Ship:


In [3]:
# Environment Classes
class RangeKeepingEnv(gym.Env):
    def __init__(self, agent_speed=1.0, schedule=None, max_steps=100000):
        super().__init__()
        self.agent_speed = agent_speed
        self.schedule = schedule if schedule else lambda t: np.array([5 + 0.1 * t, 5 + 0.1 * t])
        self.max_steps = max_steps

        # Action and Observation Spaces
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(8,), dtype=np.float32)

        # Environment State
        self.agent_pos = None
        self.opponent_pos = None

        self.agent_life = 1
        self.opponent_life = 5

        self.agent_cooldown = 20
        self.opponent_cooldown = 20

        self.agent_killcount = 0
        self.opponent_killcount = 0

        self.current_step = 0

        # Range Settings
        self.opponent_range = 5
        self.agent_range = 10

    def reset(self, seed=None, options=None):
        self.agent_pos = np.random.uniform(-20, 20, size=(2,))
        self.opponent_pos = np.random.uniform(-20, 20, size=(2,))
        self.agent_life = 1
        self.opponent_life = 5
        self.agent_cooldown = 10
        self.opponent_cooldown = 10
        self.agent_killcount = 0
        self.opponent_killcount = 0
        self.current_step = 0
        return self._get_state(), {}

    def step(self, action):
        truncated = False
        terminated = False
        self.current_step += 1
        self.agent_cooldown = max(0, self.agent_cooldown - 1)
        self.opponent_cooldown = max(0, self.opponent_cooldown - 1)

        # Normalize Action
        action = np.clip(action, -1.0, 1.0)
        self.agent_pos += action * self.agent_speed
        self.opponent_pos = self.schedule(self.current_step)

        # Calculate Distance and Rewards
        distance = np.linalg.norm(self.agent_pos - self.opponent_pos)
        reward = - distance  # Baseline penalty for every step

        if distance < self.opponent_range and self.opponent_cooldown == 0:
            self.agent_life -= 1
            self.opponent_cooldown = 10
            reward -= 1000.0  # Penalty for being hit

        if distance <= self.agent_range and self.agent_cooldown == 0:
            self.opponent_life -= 1
            self.agent_cooldown = 10
            reward += 500.0  # Reward for hitting opponent

        # Reset position if agent is dead
        if self.agent_life <= 0:
            self.agent_pos = np.random.uniform(-20, 20, size=(2,))
            self.agent_life = 1
            self.opponent_killcount += 1
        
        # Reset position if opponent is dead
        if self.opponent_life <= 0:
            self.opponent_pos = np.random.uniform(-20, 20, size=(2,))
            self.opponent_life = 5
            self.agent_killcount += 1

        # Episode truncated if max steps reached or agent out of bounds
        if self.agent_pos[0] < -25 or self.agent_pos[0] > 25 or self.agent_pos[1] < -25 or self.agent_pos[1] > 25:
            truncated = True

        if self.current_step >= self.max_steps:
            terminated = True

        return self._get_state(), reward, terminated, truncated, {}

    def render(self, mode="human"):
        print(f"Step: {self.current_step}, Agent Pos: {self.agent_pos}, Opponent Pos: {self.opponent_pos}")

    def _get_state(self):
        return np.concatenate([self.agent_pos, self.opponent_pos, [self.agent_life, self.opponent_life], [self.agent_cooldown, self.opponent_cooldown]])

In [4]:
# Movement Schedule Classes
class RandomMovementSchedule:
    def __init__(self, speed_limit=1.0, change_interval=50):
        self.speed_limit = speed_limit
        self.change_interval = change_interval
        self.current_step = 0
        self.position = np.array([5.0, 5.0])
        self._randomize_direction_and_speed()

    def __call__(self, t):
        if t % self.change_interval == 0:
            self._randomize_direction_and_speed()
        self._update_position()
        return self.position

    def _randomize_direction_and_speed(self):
        self.direction = np.random.uniform(-1, 1, size=2)
        self.direction /= np.linalg.norm(self.direction)
        self.speed = np.random.uniform(0, self.speed_limit)

    def _update_position(self):
        if self.position[0] < -20 or self.position[0] > 20:
            self.direction[0] = -self.direction[0]
            self.position[0] = np.clip(self.position[0], -20, 20)
        if self.position[1] < -20 or self.position[1] > 20:
            self.direction[1] = -self.direction[1]        
            self.position[1] = np.clip(self.position[1], -20, 20)
        self.position += self.direction * self.speed

In [5]:
# Create Gif Function
def create_gif_from_env(env, model, fps=10, steps=100, gif_name="range_keeping_test.gif"):
    frames = []
    obs, _ = env.reset()

    for _ in range(steps):
        action, _ = model.predict(obs)
        obs, _, terminated, _, _ = env.step(action)

        fig, (ax, ax_bar) = plt.subplots(2, 1, figsize=(6, 8), gridspec_kw={'height_ratios': [3, 1]})
        ax.plot(env.opponent_pos[0], env.opponent_pos[1], 'ro', label="Opponent")
        ax.plot(env.agent_pos[0], env.agent_pos[1], 'bo', label="Agent")

        #Plot positions of agent and opponent and their ranges
        opponent_range_circle = Circle(env.opponent_pos, env.opponent_range, color='red', alpha=0.2, label="Opponent Range", linestyle='dashed')
        agent_range_circle = Circle(env.agent_pos, env.agent_range, color='blue', alpha=0.1, label="Agent Range", linestyle='dashed')
        ax.add_patch(opponent_range_circle)
        ax.add_patch(agent_range_circle)

         # Draw attack lines if attacks happen
        if env.agent_cooldown > 5 and env.current_step > 5:
            ax.plot([env.agent_pos[0], env.opponent_pos[0]], [env.agent_pos[1], env.opponent_pos[1]], 'b-')
        if env.opponent_cooldown > 5 and env.current_step > 5:
            ax.plot([env.opponent_pos[0], env.agent_pos[0]], [env.opponent_pos[1], env.agent_pos[1]], 'r-')
        
        ax.legend()
        ax.set_xlim(-25, 25)
        ax.set_ylim(-25, 25)
        ax.set_title(f"Agent vs Random - Step: {env.current_step}")

        # Plot the status bars for agent and opponent
        bar_labels = ["Agent\nKillcount", "Opponent\nKillcount", "Agent\nLife",  "Opponent\nLife", "Agent\nCooldown", "Opponent\nCooldown"]
        bar_values = [env.agent_killcount, env.opponent_killcount, env.agent_life, env.opponent_life, env.agent_cooldown, env.opponent_cooldown, ]
        ax_bar.bar(bar_labels, bar_values, color=['blue', 'red', 'blue', 'red', 'blue', 'red'])
        ax_bar.set_ylim(0, 10)
        if env.agent_killcount > 10 or env.opponent_killcount > 10:
            ax_bar.set_ylim(0, np.max([env.agent_killcount, env.opponent_killcount]))

        # Capturing the frame for the gif
        fig.canvas.draw()
        frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        frames.append(frame)

        plt.close(fig)

    imageio.mimsave(gif_name, frames, fps=10)
    print(f"GIF saved as {gif_name}")

In [6]:
# Callback log class & loss plotter
class LossLoggingCallback(BaseCallback):
    def __init__(self):
        super(LossLoggingCallback, self).__init__()
        self.losses = []

    def _on_training_start(self) -> None:
        self.losses = []

    def _on_step(self) -> bool:
        if hasattr(self.model, "logger"):
            if "loss" in self.model.logger.name_to_value:
                self.losses.append(self.model.logger.name_to_value["loss"])
        return True
    
def lossplotter(losses):
    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    ax.plot(losses, label="Loss")
    ax.set_xlabel("Training Steps")
    ax.set_ylabel("Loss")
    ax.set_title("Loss Plot")
    ax.legend()
    plt.show()

In [7]:
# Training and plotting Function
def train_agent():
    env = RangeKeepingEnv(agent_speed=1.0, schedule=RandomMovementSchedule())
    env = Monitor(env)
    
    model = PPO("MlpPolicy", env, verbose=1, tensorboard_log="./tensorboard_logs/")

    model.learn(total_timesteps=1000, tb_log_name='Agent_vs_Random_1v0')

    create_gif_from_env(env, model)

if __name__ == "__main__":
    train_agent()

Using cpu device
Wrapping the env in a DummyVecEnv.
Logging to ./tensorboard_logs/Agent_vs_Random_1v0_10
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 369       |
|    ep_rew_mean     | -7.74e+03 |
| time/              |           |
|    fps             | 3485      |
|    iterations      | 1         |
|    time_elapsed    | 0         |
|    total_timesteps | 2048      |
----------------------------------


c:\Users\micha\anaconda3\Lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.opponent_pos to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.opponent_pos` for environment variables or `env.get_wrapper_attr('opponent_pos')` that will search the reminding wrappers.
  logger.warn(
c:\Users\micha\anaconda3\Lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.agent_pos to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.agent_pos` for environment variables or `env.get_wrapper_attr('agent_pos')` that will search the reminding wrappers.
  logger.warn(
c:\Users\micha\anaconda3\Lib\site-packages\gymnasium\core.py:311: UserWarning: WARN: env.opponent_range to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.opponent_range` for environment variables or

GIF saved as range_keeping_test.gif
